# Task 3: Partition Pruning Experiment (Queries 1, 6)

- February 2024 queries are run on a month-partitioned Gold table and an identical unpartitioned Delta copy.

## Connect to Spark

In [1]:
import sys

from pyspark.sql import SparkSession

JOB_PATH = '/opt/spark/jobs'
if JOB_PATH not in sys.path:
    sys.path.insert(0, JOB_PATH)

from analytics.benchmarking import measure_registered_query, rows_match
from analytics.runner import (
    DEFAULT_INTEGRATED_TABLE_PATH,
    INTEGRATED_TABLE_VIEW,
    execute_registered_query,
    register_integrated_taxi_trips,
)

spark = (
    SparkSession.builder
    .master('spark://spark-master:7077')
    .appName('Assignment2-PartitionPruning-Query1-6')
    .getOrCreate()
)
spark.conf.set('spark.sql.session.timeZone', 'UTC')
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/20 18:10:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Define the two layouts

- February is used because the data covers January-March 2024. 
- Partition prunnig by skipping January and March.

In [2]:
PARTITIONED_TABLE_PATH = DEFAULT_INTEGRATED_TABLE_PATH
UNPARTITIONED_TABLE_PATH = (
    's3a://delta-data/gold/gold_integrated_taxi_trips_unpartitioned'
)
QUERY_NAMES = (
    'query_01_monthly_zone_demand_february_2024',
    'query_06_monthly_taxi_demand_trend_february_2024',
)
REPETITIONS = 9

## Create the unpartitioned layout

In [3]:
partitioned_table = spark.read.format('delta').load(PARTITIONED_TABLE_PATH)

(
    partitioned_table.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .save(UNPARTITIONED_TABLE_PATH)
)

unpartitioned_table = (
    spark.read.format('delta').load(UNPARTITIONED_TABLE_PATH)
)

assert partitioned_table.count() == unpartitioned_table.count()
print('Unpartitioned layout created with the same number of rows.')

26/09/20 18:10:34 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/09/20 18:10:37 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Unpartitioned layout created with the same number of rows.


## Measure both layouts

Each layout receives one warm-up followed by three timed runs.

In [4]:
def measure_layout(table_path):
    spark.catalog.clearCache()
    register_integrated_taxi_trips(spark, table_path)
    return {
        query_name: measure_registered_query(spark, query_name, REPETITIONS)
        for query_name in QUERY_NAMES
    }

partitioned_results = measure_layout(PARTITIONED_TABLE_PATH)
unpartitioned_results = measure_layout(UNPARTITIONED_TABLE_PATH)

for query_name in QUERY_NAMES:
    assert rows_match(
        partitioned_results[query_name]['rows'],
        unpartitioned_results[query_name]['rows'],
    ), f'Partitioned and unpartitioned results differ for {query_name}.'

print('Both layouts return the same results within float tolerance.')

Both layouts return the same results within float tolerance.


## Partitioned physical plans

In [5]:
register_integrated_taxi_trips(spark, PARTITIONED_TABLE_PATH)
for query_name in QUERY_NAMES:
    print(f'\n--- {query_name} ---')
    execute_registered_query(spark, query_name).explain('formatted')


--- query_01_monthly_zone_demand_february_2024 ---
== Physical Plan ==
AdaptiveSparkPlan (7)
+- Sort (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Exchange (3)
            +- HashAggregate (2)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [pickup_zone_name#7090, pickup_month#7084]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/gold/gold_integrated_taxi_trips]
PartitionFilters: [isnotnull(pickup_month#7084), (pickup_month#7084 = 2024-02)]
ReadSchema: struct<pickup_zone_name:string>

(2) HashAggregate
Input [2]: [pickup_zone_name#7090, pickup_month#7084]
Keys [2]: [pickup_zone_name#7090, pickup_month#7084]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#7257L]
Results [3]: [pickup_zone_name#7090, pickup_month#7084, count#7258L]

(3) Exchange
Input [3]: [pickup_zone_name#7090, pickup_month#7084, count#7258L]
Arguments: hashpartitioning(pickup_zone_name#7090, pickup_month#7084, 4), ENSURE_REQUIREMENTS, [plan_id=454

## Unpartitioned physical plans

- It does not contain a `PartitionFilters` entry.

In [6]:
register_integrated_taxi_trips(spark, UNPARTITIONED_TABLE_PATH)
for query_name in QUERY_NAMES:
    print(f'\n--- {query_name} ---')
    execute_registered_query(spark, query_name).explain('formatted')


--- query_01_monthly_zone_demand_february_2024 ---
== Physical Plan ==
AdaptiveSparkPlan (8)
+- Sort (7)
   +- Exchange (6)
      +- HashAggregate (5)
         +- Exchange (4)
            +- HashAggregate (3)
               +- Filter (2)
                  +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [pickup_month#7412, pickup_zone_name#7418]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/gold/gold_integrated_taxi_trips_unpartitioned]
PushedFilters: [IsNotNull(pickup_month), EqualTo(pickup_month,2024-02)]
ReadSchema: struct<pickup_month:string,pickup_zone_name:string>

(2) Filter
Input [2]: [pickup_month#7412, pickup_zone_name#7418]
Condition : (isnotnull(pickup_month#7412) AND (pickup_month#7412 = 2024-02))

(3) HashAggregate
Input [2]: [pickup_month#7412, pickup_zone_name#7418]
Keys [2]: [pickup_zone_name#7418, pickup_month#7412]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#7581L]
Results [3]: [pickup_zone_name#7418, pickup_month#7412, 

## Benchmark summary

In [7]:
summary = []
for query_name in QUERY_NAMES:
    partitioned = partitioned_results[query_name]
    unpartitioned = unpartitioned_results[query_name]
    summary.append({
        'query': query_name,
        'partitioned_samples_seconds': [
            round(value, 3) for value in partitioned['samples_seconds']
        ],
        'partitioned_median_seconds': round(partitioned['median_seconds'], 3),
        'unpartitioned_samples_seconds': [
            round(value, 3) for value in unpartitioned['samples_seconds']
        ],
        'unpartitioned_median_seconds': round(unpartitioned['median_seconds'], 3),
        'partitioned_speedup_factor': round(
            unpartitioned['median_seconds'] / partitioned['median_seconds'],
            2,
        ),
    })
summary

[{'query': 'query_01_monthly_zone_demand_february_2024',
  'partitioned_samples_seconds': [1.531,
   0.955,
   0.818,
   1.017,
   0.766,
   1.014,
   1.11,
   0.776,
   0.762],
  'partitioned_median_seconds': 0.955,
  'unpartitioned_samples_seconds': [1.526,
   1.086,
   0.825,
   0.838,
   0.88,
   0.935,
   0.889,
   0.876,
   1.098],
  'unpartitioned_median_seconds': 0.889,
  'partitioned_speedup_factor': 0.93},
 {'query': 'query_06_monthly_taxi_demand_trend_february_2024',
  'partitioned_samples_seconds': [1.065,
   0.672,
   0.741,
   0.595,
   0.847,
   0.664,
   0.627,
   0.616,
   0.862],
  'partitioned_median_seconds': 0.672,
  'unpartitioned_samples_seconds': [0.69,
   0.579,
   0.736,
   0.657,
   0.67,
   0.903,
   0.695,
   0.741,
   0.653],
  'unpartitioned_median_seconds': 0.69,
  'partitioned_speedup_factor': 1.03}]

## Cleanup

In [ ]:
spark.catalog.clearCache()
spark.stop()

26/09/20 14:03:58 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-dbb289fa-90f5-4917-b9ca-66aedffc4088/pyspark-c0d1395d-9354-4b64-8ab4-c4dc6b086998. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-dbb289fa-90f5-4917-b9ca-66aedffc4088/pyspark-c0d1395d-9354-4b64-8ab4-c4dc6b086998
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:191)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:116)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:94)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1048)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4(ShutdownHookManager.scala:70)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun